# Slim down PR #74 to the fine-tuning setup only
Updates the **existing** branch `feature/router-study`, so #74 updates in place:
- **removes** the results, charts, notebook and training log, and the changes to existing files (README and the three placeholder scripts go back to exactly what `dev` has);
- **adds** `requirements-research.txt`, a short `fine_tuning/README_router_study.md`, and a leakage test (no validation or final question in the training data);
- fixes the chart test so it skips without matplotlib.

Afterwards #74 is 8 new files and nothing else. Needs `GITHUB_PAT` and **`router_study_slim.zip`**.

## 1. Authenticate and clone the PR branch

In [ ]:
import os, subprocess, sys
from pathlib import Path
from google.colab import userdata
github_token = userdata.get("GITHUB_PAT")
if not github_token:
    raise RuntimeError("Colab Secret 'GITHUB_PAT' is empty or missing.")
os.environ["GH_TOKEN"] = github_token
account = subprocess.run(["gh", "api", "user", "--jq", ".login"], capture_output=True, text=True, check=True).stdout.strip()
os.chdir("/content")
REPO_ROOT = Path("/content/woodb-agent-project"); BRANCH = "feature/router-study"
subprocess.run(["gh", "auth", "setup-git"], check=True)
subprocess.run(["rm", "-rf", str(REPO_ROOT)], check=True)
subprocess.run(["git", "clone", "--quiet", "--branch", BRANCH, "https://github.com/JP-DSmith/woodb-agent-project.git", str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
subprocess.run(["git", "config", "user.name", account], check=True)
subprocess.run(["git", "config", "user.email", f"{account}@users.noreply.github.com"], check=True)
subprocess.run(["git", "fetch", "--quiet", "origin", "dev"], check=True)
print("Signed in as", account); print(subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)

## 2. Remove what isn't needed, restore existing files to `dev`

In [ ]:
subprocess.run(["git", "rm", "-rq", "fine_tuning/results/router_study", "fine_tuning/backup/router_study.ipynb"], check=True)
subprocess.run(["git", "checkout", "origin/dev", "--", "README.md", "fine_tuning/train_small_model.py",
                "fine_tuning/prepare_data.py", "evaluation/evaluate_models.py"], check=True)
print(subprocess.run(["git", "status", "--short"], capture_output=True, text=True).stdout)

## 3. Upload `router_study_slim.zip`

In [ ]:
import zipfile
from google.colab import files
EXPECTED = sorted(["requirements-research.txt", "testing/test_backup_router.py", "fine_tuning/README_router_study.md"])
uploaded = files.upload()
zip_name = next((n for n in uploaded if n.startswith("router_study_slim") and n.endswith(".zip")), None)
if not zip_name:
    raise RuntimeError("Please upload router_study_slim.zip")
with zipfile.ZipFile(zip_name) as z:
    names = sorted(n for n in z.namelist() if not n.endswith("/"))
    if names != EXPECTED:
        raise RuntimeError(f"Unexpected files in the zip: {names}")
    z.extractall(REPO_ROOT)
os.remove(zip_name)
subprocess.run(["git", "add", "-A"], check=True)
changed = subprocess.run(["git", "diff", "--cached", "--name-status", "origin/dev"], capture_output=True, text=True).stdout
print("Compared with dev, the PR will contain:\n" + changed)
if any(not line.startswith("A") for line in changed.splitlines() if line.strip()):
    raise RuntimeError("Something other than new files differs from dev: stop and send this output.")

## 4. Tests as in CI (no research packages) and `git diff --check`

In [ ]:
subprocess.run(["pip", "-q", "install", "-r", "requirements.txt", "pyyaml", "pytest"], check=True)
failed = []
for name, args in [("study tests", ["testing/test_backup_router.py"]), ("full suite", [])]:
    run = subprocess.run([sys.executable, "-m", "pytest", "-q", *args], capture_output=True, text=True)
    print(f"{name}: {run.stdout.strip().splitlines()[-1] if run.stdout.strip() else run.stderr[-300:]}")
    if run.returncode:
        print(run.stdout[-1500:]); failed.append(name)
check = subprocess.run(["git", "diff", "--cached", "--check"], capture_output=True, text=True)
print("git diff --check:", "clean" if not check.returncode else check.stdout[:1500])
if failed or check.returncode:
    raise RuntimeError("Something failed: don't push. Send the output above.")

## 5. Commit, push (PR #74 updates), update the PR description and reply

In [ ]:
import base64
msg = ("refactor: slim the router study PR to the fine-tuning setup only\n\n"
       "Removes the committed results, charts, notebook and training log (results go in the report) and the\n"
       "changes to existing files. Adds requirements-research.txt, a short README, and a test that no validation\n"
       "or final question (holdout, team, hand-written, attacks) appears in the training data. The chart test\n"
       "skips without matplotlib.\n\n"
       "Co-Authored-By: Claude Opus 5.5 <noreply@anthropic.com>\n"
       "Claude-Session: https://claude.ai/code/session_01V8VAWbrtAqnf7y3QSxGDSR")
subprocess.run(["git", "commit", "-m", msg], check=True)
auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
push = subprocess.run(["git", "-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {auth}", "push", "origin", BRANCH],
                      capture_output=True, text=True)
if push.returncode:
    print(push.stderr); raise RuntimeError("Push failed.")
body = """## What this adds
The router fine-tuning setup (brief section 2), and nothing else: **8 new files, no existing file changed.**

- `fine_tuning/backup/`: training (`train_finetuned.py`), the settings search and controlled versions (`experiments.py`), the routers compared (`routers.py`), metrics and statistics (`analysis.py`), and the economics scenarios (`economics.py`).
- `fine_tuning/README_router_study.md`: what each file does, the data rules, how to install and run it.
- `requirements-research.txt`: the extra packages for training, analysis and charts (not needed by the app).
- `testing/test_backup_router.py`: 14 tests. In normal CI the data, prompt, leakage, metrics and economics tests run, and the training, chart and ONNX tests skip; with `pip install -r requirements.txt -r requirements-research.txt`, all 14 run.

No results, charts or notebook are committed: the results and conclusion are in the report (§5 and Appendix D). The economics are conditional scenarios on stated assumptions, as the README says. This PR is separate from the production router decision (#72).
"""
subprocess.run(["gh", "pr", "edit", "74", "--repo", "JP-DSmith/woodb-agent-project", "--body",
                body + "\n\n🤖 Generated with [Claude Code](https://claude.com/claude-code)\n\nhttps://claude.ai/code/session_01V8VAWbrtAqnf7y3QSxGDSR"], check=True)
reply = """Thanks for the thorough review. I've slimmed this PR to the setup only, which is what the brief needs in the repository: 8 new files, no existing file changed, no results committed (they go in the report).

- Items 1, 2 and 6 (raw rows, run manifest, report reproduction): no results, charts or notebook are committed any more, so there's nothing in the repo making claims that would need them.
- Item 3: a new test checks that no validation or final question (holdout seen and new, team AI and hand-written, human-written, and the attack suite) appears in the training data; few-shot examples come from training only (existing test).
- Items 4 and 5: `requirements-research.txt` plus install instructions in the README; the test file says which tests run in normal CI and which need the research packages. The chart test now skips without matplotlib. With the research packages installed, all 14 tests pass.
- Item 7: the README states the economics are conditional scenarios on stated assumptions.
- Item 8: agreed, separate from #72. The supplier-question regression you found is fixed on #72 (all 15 supplier questions, including your 5 Portal cases, route to sourcing_whatif with the real model; regression tests and evidence committed there)."""
subprocess.run(["gh", "pr", "comment", "74", "--repo", "JP-DSmith/woodb-agent-project", "--body", reply], check=True)
print("Pushed; #74 is slimmed, its description updated and the reply posted.")